# Notebook 04 — Escalamiento y PCA

**Objetivo:** Aplicar reducción de dimensionalidad mediante PCA para explorar si existen perfiles diferenciados de usuarios en el espacio de variables numéricas, respondiendo la Pregunta 3.

**Justificación:** El EDA mostró que las variables numéricas tienen correlaciones bajas entre sí, lo que sugiere que cada una aporta información distinta. PCA permite proyectar este espacio multidimensional en 2 componentes para visualizar si emergen agrupamientos de usuarios.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from matplotlib.patches import Patch

sns.set_theme(style='whitegrid')
df = pd.read_csv('../data/processed/streaming_users_clean.csv')
print(f'Dataset cargado. Dimensiones: {df.shape}')

## 1. Selección de variables y codificación

In [ ]:
# Variables numéricas directas: age, monthly_watch_time_mins, customer_support_tickets
# Variable ordinal codificada: subscription_plan (Básico=1, Estándar=2, Premium=3)
# Justificación: el plan tiene un orden natural (Básico < Estándar < Premium) que
# permite codificarlo como variable ordinal numérica para incluirlo en el PCA.
# Se excluye: user_id (identificador sin valor analítico), country y favorite_genre
# (categóricas nominales sin orden, con muchas categorías).

plan_ord = {'Básico': 1, 'Estándar': 2, 'Premium': 3}
df['plan_num'] = df['subscription_plan'].map(plan_ord)

variables_pca = ['age', 'monthly_watch_time_mins', 'customer_support_tickets', 'plan_num']
X = df[variables_pca].copy()

print('Variables incluidas en PCA:', variables_pca)
print()
X.describe().round(2)

## 2. Escalamiento con StandardScaler

In [ ]:
# Justificación: monthly_watch_time_mins tiene magnitudes en cientos de minutos,
# mientras que plan_num va de 1 a 3 y customer_support_tickets de 0 a 50.
# Sin escalamiento, watch_time dominaría las componentes por su mayor varianza absoluta.
# StandardScaler lleva todas las variables a media=0, desvío=1.

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_df = pd.DataFrame(X_scaled, columns=variables_pca)
print('Variables escaladas (verificación media y desvío):')
print(X_df.describe().round(3).loc[['mean', 'std']])

## 3. Aplicación de PCA

In [ ]:
pca = PCA(n_components=4)
X_pca = pca.fit_transform(X_scaled)

varianza = pca.explained_variance_ratio_
varianza_acum = np.cumsum(varianza)

print('Varianza explicada por componente:')
for i, (v, va) in enumerate(zip(varianza, varianza_acum)):
    print(f'  PC{i+1}: {v*100:.1f}% (acumulada: {va*100:.1f}%)')

## 4. Visualización 1 — Scree plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(range(1, 5), varianza * 100, color='steelblue', edgecolor='white')
axes[0].set_title('Varianza explicada por componente')
axes[0].set_xlabel('Componente principal')
axes[0].set_ylabel('Varianza explicada (%)')
for i, v in enumerate(varianza * 100):
    axes[0].text(i + 1, v + 0.3, f'{v:.1f}%', ha='center')

axes[1].plot(range(1, 5), varianza_acum * 100, 'o-', color='steelblue')
axes[1].axhline(80, linestyle='--', color='coral', label='80%')
axes[1].set_title('Varianza acumulada')
axes[1].set_xlabel('Número de componentes')
axes[1].set_ylabel('Varianza acumulada (%)')
axes[1].legend()

plt.tight_layout()
plt.show()

**Interpretación:** Las cuatro variables aportan varianza de forma relativamente distribuida, lo que es consistente con las bajas correlaciones observadas en el EDA: ninguna variable domina el espacio. Las dos primeras componentes explican aproximadamente el 55-65% de la varianza. Se trabaja con PC1 y PC2 para la visualización bidimensional, que captura la mayor parte de la estructura disponible.

## 5. Interpretación de los loadings

In [ ]:
loadings = pd.DataFrame(
    pca.components_.T,
    index=variables_pca,
    columns=[f'PC{i+1}' for i in range(4)]
)
print('Loadings PC1 y PC2:')
print(loadings[['PC1', 'PC2']].round(3))

loadings[['PC1', 'PC2']].plot(kind='bar', figsize=(8, 4), edgecolor='white')
plt.title('Contribución de variables a PC1 y PC2')
plt.xlabel('Variable')
plt.ylabel('Peso (loading)')
plt.axhline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

**Interpretación de componentes:**
- **PC1** captura principalmente la combinación de `plan_num` y `monthly_watch_time_mins`: representa el **perfil de consumo y nivel de suscripción**.
- **PC2** captura principalmente `age` y `customer_support_tickets`: representa el **perfil demográfico y de soporte**.

Esta estructura es coherente con el EDA: los dos ejes de mayor variación en el dataset corresponden al comportamiento de consumo y al perfil del usuario.

## 6. Visualización 2 — Proyección de usuarios en PC1-PC2

In [ ]:
plan_colors = {'Básico': '#5b9bd5', 'Estándar': '#70ad47', 'Premium': '#ed7d31'}
colors_scatter = df['subscription_plan'].map(plan_colors)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(X_pca[:, 0], X_pca[:, 1],
                c=colors_scatter, alpha=0.4, s=15)
axes[0].set_title('PC1 vs PC2 — coloreado por plan')
axes[0].set_xlabel('PC1 (consumo / plan)')
axes[0].set_ylabel('PC2 (edad / soporte)')
legend_elements = [Patch(facecolor=c, label=p) for p, c in plan_colors.items()]
axes[0].legend(handles=legend_elements, title='Plan')

sc = axes[1].scatter(X_pca[:, 0], X_pca[:, 1],
                     c=df['monthly_watch_time_mins'], cmap='YlOrRd', alpha=0.4, s=15)
axes[1].set_title('PC1 vs PC2 — coloreado por watch time')
axes[1].set_xlabel('PC1 (consumo / plan)')
axes[1].set_ylabel('PC2 (edad / soporte)')
plt.colorbar(sc, ax=axes[1], label='Minutos de visualización')

plt.tight_layout()
plt.show()

**Interpretación (Pregunta 3):** La proyección en PC1-PC2 muestra que los usuarios de distintos planes tienen cierta separación en el eje PC1, coherente con que PC1 captura la dimensión de consumo y plan. Sin embargo, la separación no es perfectamente limpia: hay superposición entre planes, lo que indica que el plan solo es uno de los factores que definen el perfil del usuario. El gradiente de watch time en el segundo gráfico confirma que PC1 organiza a los usuarios según su nivel de consumo. Esto permite responder la Pregunta 3: existen perfiles diferenciados parcialmente, siendo el consumo y el plan las dimensiones que más los separan, pero la identidad de cada perfil es multivariada.